# Fama-French Regressions by Year and Decile
Este notebook recalcula as regressões de Fama-French de 3 Fatores para as carteiras (decis) formadas pelas métricas de rede (HRM e Pozzi).
A lógica atual utiliza os ativos agrupados em 10 decis para cada ano, construídos _In-Sample_ (ano $t$), e avaliados _Out-of-Sample_ (ano $t+1$).\n

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

## 1. Processamento e Regressões OLS
Aqui nós iteramos pelos anos (2014 a 2024), carregamos as rentabilidades diárias do ano seguinte, filtramos os tickers de cada decil e rodamos a regressão OLS.
Usamos o cálculo de **Equal-Weighted** log returns para os portfólios, garantindo que a carteira capte o efeito puramente estrutural da rede (sem enviesar por Market Cap).\n

In [ ]:
############### BUY N HOLD 1/N ####################

import pandas as pd
import numpy as np
import statsmodels.api as sm

# Carrega os fatores de Fama-French
factors_df = pd.read_parquet("../../data/02_clean/fama_french_factors.parquet")

# Os fatores originais geralmente vêm em porcentagem (ex: 1.5%), então dividimos por 100
factors_df = factors_df / 100

years = range(2014, 2025)
metrics = ['hcm', 'pozzi'] 
deciles = [f'decil_{i}' for i in range(1, 11)]

# Dicionário para armazenar resultados puros das regressões
regression_results = []

for metric in metrics:
    print(f"Processando regressões para: {metric.upper()}...")
    
    # Carrega metadados que dizem qual Ticker está em qual decil a cada ano
    try:
        df_meta = pd.read_parquet(f"../../data/07_portfolios_metadata/complete_metadata_{metric}.parquet")
        df_meta['year'] = df_meta['year'].astype(int)
    except FileNotFoundError:
        print(f"Arquivo de metadados para {metric} não encontrado. Pule.")
        continue
    
    for year in years:
        # Puxa retornos Out-of-Sample (ano + 1)
        try:
            oos_ret = pd.read_parquet(f"../../data/02_clean/returns_new_{year+1}.parquet")
        except FileNotFoundError:
            continue
            
        # Filtro de segurança contra outlier severo que destrói o Buy-and-Hold
        if 'HYFT' in oos_ret.columns:
            oos_ret = oos_ret.drop(columns=['HYFT'])
            
        # Alinha os fatores de Fama-French às datas dos retornos daquele ano
        factors_year = factors_df[(factors_df.index >= oos_ret.index[0]) & (factors_df.index <= oos_ret.index[-1])]
        
        # Dicionário temporário para guardar os retornos das pontas do PMC
        port_returns_year = {}
        
        # =========================================================
        # 1. Regressões individuais para cada decil
        # =========================================================
        for decil in deciles:
            # Puxa tickers daquele decil no ano 'year'
            tickers = df_meta[(df_meta['year'] == year) & (df_meta['portfolio'] == decil)]['Ticker'].tolist()
            
            # Garante que as ações existam na base de retornos do ano t+1
            valid_tickers = [t for t in tickers if t in oos_ret.columns]
            
            if len(valid_tickers) == 0:
                continue
                
            # --- CÁLCULO BUY-AND-HOLD 1/N ---
            df_ret_filled = oos_ret[valid_tickers].fillna(0)
            cum_ret_ativos = (1 + df_ret_filled).cumprod()
            port_cum_ret = cum_ret_ativos.mean(axis=1)
            
            # Extrai o retorno diário SIMPLES da variação do portfólio
            port_ret = port_cum_ret.pct_change()
            port_ret.iloc[0] = port_cum_ret.iloc[0] - 1
            # --------------------------------
            
            # Salva para criar o PMC depois
            if decil in ['decil_1', 'decil_10']:
                port_returns_year[decil] = port_ret
            
            # Subtrai a Risk-Free rate para obter o Excesso de Retorno
            excess_ret = port_ret - factors_year['RF']
            
            # Variáveis independentes
            X = factors_year[['Mkt-RF', 'SMB', 'HML']]
            X = sm.add_constant(X)
            
            # Alinhamento por data e drop de NAs
            aligned = pd.concat([excess_ret.rename("ExRet"), X], axis=1, join="inner").dropna()
            
            if len(aligned) < 30:
                continue
                
            # Roda a Regressão
            model = sm.OLS(aligned['ExRet'], aligned[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
            
            regression_results.append({
                'metric': metric,
                'year_oos': year + 1,
                'year_formed': year,
                'decil': decil,
                'alpha': model.params['const'] * 252, # Alpha Anualizado
                'alpha_tstat': model.tvalues['const'],
                'mkt_beta': model.params['Mkt-RF'],
                'smb_beta': model.params['SMB'],
                'hml_beta': model.params['HML'],
                'r_squared': model.rsquared
            })
            
        # =========================================================
        # 2. Regressão do Portfólio PMC (Peripheral Minus Central)
        # =========================================================
        if 'decil_10' in port_returns_year and 'decil_1' in port_returns_year:
            # Retorno simples do PMC = Peripheral(decil_10) - Central(decil_1)
            pmc_ret = port_returns_year['decil_10'] - port_returns_year['decil_1']
            
            # NOTA: O fator RF se anula em carteiras Long-Short, 
            # portanto o retorno líquido pmc_ret JÁ É O EXCESSO DE RETORNO!
            
            X = factors_year[['Mkt-RF', 'SMB', 'HML']]
            X = sm.add_constant(X)
            
            aligned_pmc = pd.concat([pmc_ret.rename("ExRet"), X], axis=1, join="inner").dropna()
            
            if len(aligned_pmc) >= 30:
                model_pmc = sm.OLS(aligned_pmc['ExRet'], aligned_pmc[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
                
                regression_results.append({
                    'metric': metric,
                    'year_oos': year + 1,
                    'year_formed': year,
                    'decil': 'PMC (10 - 1)', # Identificador único para essa carteira
                    'alpha': model_pmc.params['const'] * 252,
                    'alpha_tstat': model_pmc.tvalues['const'],
                    'mkt_beta': model_pmc.params['Mkt-RF'],
                    'smb_beta': model_pmc.params['SMB'],
                    'hml_beta': model_pmc.params['HML'],
                    'r_squared': model_pmc.rsquared
                })

df_results = pd.DataFrame(regression_results)
print("Todas as regressões foram calculadas, usando Buy-and-Hold 1/N!")

Processando regressões para: HCM...
Processando regressões para: POZZI...
Todas as regressões foram calculadas, usando Buy-and-Hold 1/N!


## 2. Opção de Tabela: Resumo Estilo Fama-MacBeth
Esta tabela sintetiza 10 anos de regressões. Ela exibe a **média temporal** dos Alphas e dos Betas para cada decil. 
O teste t (Alpha_tstat_FM) é calculado dividindo a média do Alpha pelo Erro Padrão da média (Desvio Padrão do Alpha / raiz do número de anos). Essa é a forma mais clássica de relatar performance em asset pricing.\n

In [ ]:
def generate_fama_macbeth_table(df_metric_results, metric_name):
    if df_metric_results.empty:
        return None
        
    # Número de anos na amostra OOS
    n_years = df_metric_results['year_oos'].nunique()
    
    # Agrega tirando a média das variáveis e o desvio padrão do alpha
    # Como já incluímos o "PMC (10 - 1)" nas regressões originais, 
    # ele já entra neste cálculo automático junto com os decis!
    summary = df_metric_results.groupby('decil').agg({
        'alpha': ['mean', 'std'],
        'mkt_beta': 'mean',
        'smb_beta': 'mean',
        'hml_beta': 'mean',
        'r_squared': 'mean'
    })
    
    # Calcula T-Statistic no estilo Fama-MacBeth (Mean / Standard Error)
    summary['Alpha_tstat_FM'] = summary[('alpha', 'mean')] / (summary[('alpha', 'std')] / np.sqrt(n_years))
    
    # Achata as colunas (Flatten)
    summary.columns = ['Alpha', 'Alpha_Std', 'Mkt_Beta', 'SMB_Beta', 'HML_Beta', 'R_Squared', 'Alpha_tstat_FM']
    
    # Reordena para ficar bonito (decil_1 até decil_10 e depois a carteira Long-Short PMC)
    decil_order = [f'decil_{i}' for i in range(1, 11)]
    if 'PMC (10 - 1)' in summary.index:
        decil_order.append('PMC (10 - 1)')
    
    # Filtra e aplica a ordem correta
    summary = summary.reindex(decil_order)
    
    # Formata para visualização
    final_table = summary[['Alpha', 'Alpha_tstat_FM', 'Mkt_Beta', 'SMB_Beta', 'HML_Beta', 'R_Squared']].copy()
    final_table = final_table.round(4)
    
    # Renomeia o índice linha por linha com base no nome original
    index_names = []
    for idx in final_table.index:
        if idx == 'decil_1':
            index_names.append('Decil 1 (Central)')
        elif idx == 'decil_10':
            index_names.append('Decil 10 (Peripheral)')
        elif idx == 'PMC (10 - 1)':
            index_names.append('Long-Short (Peripheral - Central)')
        else:
            index_names.append(idx.replace('_', ' ').title())
            
    final_table.index = index_names
    
    print(f"\n== FAMA-MACBETH REGRESSION SUMMARY: {metric_name.upper()} ===")
    display(final_table)
    
    return final_table

# Executa para HCM e Pozzi (lembrando que atualizamos a nomenclatura!)
fm_hcm = generate_fama_macbeth_table(df_results[df_results['metric'] == 'hcm'], 'hcm')
fm_pozzi = generate_fama_macbeth_table(df_results[df_results['metric'] == 'pozzi'], 'pozzi')

# Salva tabelas
if fm_hcm is not None:
    fm_hcm.to_csv("../../data/07_portfolios_metadata/table_famamacbeth_hcm.csv")
if fm_pozzi is not None:
    fm_pozzi.to_csv("../../data/07_portfolios_metadata/table_famamacbeth_pozzi.csv")


== FAMA-MACBETH REGRESSION SUMMARY: HCM ===


,Alpha,Alpha_tstat_FM,Mkt_Beta,SMB_Beta,HML_Beta,R_Squared
Decil 1 (Central),-0.0240,-2.2673,0.8915,0.5799,0.3544,0.9537
Decil 2,-0.0003,-0.0287,0.9349,0.5425,0.2929,0.9685
Decil 3,-0.0102,-0.8597,0.8809,0.5041,0.3126,0.9689
Decil 4,-0.0159,-1.2908,0.8718,0.4681,0.2849,0.9632
Decil 5,-0.0089,-0.9992,0.8138,0.4344,0.2186,0.9407
Decil 6,0.0038,0.3295,0.8196,0.4000,0.1749,0.9160
Decil 7,0.0172,1.3103,0.7858,0.3582,0.0605,0.8672
Decil 8,-0.0085,-0.5107,0.7017,0.3867,0.0453,0.8352
Decil 9,0.0298,0.9748,0.5516,0.4517,0.0088,0.6467
Decil 10 (Peripheral),0.1013,2.0682,0.2808,0.2452,-0.0415,0.3562



== FAMA-MACBETH REGRESSION SUMMARY: POZZI ===


,Alpha,Alpha_tstat_FM,Mkt_Beta,SMB_Beta,HML_Beta,R_Squared
Decil 1 (Central),-0.0085,-0.5248,0.7816,0.2539,0.3019,0.8617
Decil 2,0.0069,0.5295,0.7619,0.3030,0.2713,0.8620
Decil 3,-0.0138,-1.1350,0.7907,0.3741,0.2149,0.9238
Decil 4,0.0223,1.9243,0.8219,0.4224,0.1668,0.8965
Decil 5,-0.0064,-0.3805,0.8112,0.4686,0.1640,0.8756
Decil 6,0.0080,0.5255,0.7737,0.5012,0.1385,0.8820
Decil 7,0.0106,0.7550,0.7917,0.5691,0.1517,0.8822
Decil 8,-0.0124,-0.7334,0.7866,0.5284,0.1618,0.8716
Decil 9,0.0159,0.4484,0.6374,0.4312,0.0860,0.7477
Decil 10 (Peripheral),0.0645,1.8087,0.5528,0.5139,0.0326,0.5839


## Por DECIS

In [9]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from IPython.display import display

# =========================================================================
# CONFIGURAÇÃO E CARREGAMENTO
# =========================================================================
years = range(2014, 2025)
metrics = ['hcm', 'pozzi'] 
decile_pairs = [(10,1), (9,2), (8,3), (7,4), (6,5)] 
c_tc = 0.001 # Custo de transação (10 bps)

# Fatores Fama-French
factors_df = pd.read_parquet("../../data/02_clean/fama_french_factors.parquet") / 100

# Retornos base
df_ret_full = pd.read_parquet("../../data/01_raw/returns.parquet")
df_ret_full = df_ret_full[df_ret_full.index >= pd.Timestamp(2015, 1, 1)]

if 'HYFT' in df_ret_full.columns:
    df_ret_full = df_ret_full.drop(columns=['HYFT'])

# Dicionários para manter a memória dos pesos finais (w_last)
# Usa a chave exatamente na ordem do decile_pairs
w_last_c = {m: {f"d{c}_d{p}": pd.Series(dtype=float) for c, p in decile_pairs} for m in metrics}
w_last_p = {m: {f"d{c}_d{p}": pd.Series(dtype=float) for c, p in decile_pairs} for m in metrics}
w_last_union = {m: {f"d{c}_d{p}": pd.Series(dtype=float) for c, p in decile_pairs} for m in metrics}

regression_results = []

def calc_buy_hold_return_with_tc(valid_tickers, df_returns, w_last):
    """Calcula o retorno B&H, deduz custo de transação e devolve os pesos pro próximo ano"""
    if len(valid_tickers) == 0:
        return None, w_last
        
    df_ret_filled = df_returns[valid_tickers].fillna(0)
    cum_ret_ativos = (1 + df_ret_filled).cumprod()
    port_cum_ret = cum_ret_ativos.mean(axis=1) # Começa igual 1/N
    
    ret = port_cum_ret.pct_change()
    ret.iloc[0] = port_cum_ret.iloc[0] - 1
    
    # Turnover e TC
    w_new = pd.Series(1.0 / len(valid_tickers), index=valid_tickers)
    turnover = w_new.sub(w_last, fill_value=0).abs().sum()
    tc = c_tc * turnover
    
    # Subtrai o custo no primeiro dia útil
    ret.iloc[0] = ret.iloc[0] - tc
    
    # Atualiza w_last para o fim do ano
    new_w_last = cum_ret_ativos.iloc[-1] / cum_ret_ativos.iloc[-1].sum()
    
    return ret, new_w_last

# =========================================================================
# CÁLCULO DE ESTRATÉGIAS E REGRESSÕES ANUAIS
# =========================================================================
for metric in metrics:
    print(f"Calculando {metric.upper()}...")
    
    try:
        df_meta = pd.read_parquet(f"../../data/07_portfolios_metadata/complete_metadata_{metric}.parquet")
        df_meta['year'] = df_meta['year'].astype(int)
    except FileNotFoundError:
        continue
    
    for year in years:
        test_year = year + 1
        oos_ret = df_ret_full[df_ret_full.index.year == test_year]
        if oos_ret.empty: continue
            
        factors_year = factors_df[(factors_df.index >= oos_ret.index[0]) & (factors_df.index <= oos_ret.index[-1])]
        X_base = sm.add_constant(factors_year[['Mkt-RF', 'SMB', 'HML']])
        
        for c_idx, p_idx in decile_pairs:
            chave_par = f"D{c_idx}_D{p_idx}"
            chave_dict = f"d{c_idx}_d{p_idx}" # Padronizado pra não dar key error
            
            c_label, p_label = f"decil_{c_idx}", f"decil_{p_idx}"
            
            tickers_c = df_meta[(df_meta['year'] == year) & (df_meta['portfolio'] == c_label)]['Ticker'].tolist()
            tickers_p = df_meta[(df_meta['year'] == year) & (df_meta['portfolio'] == p_label)]['Ticker'].tolist()
            tickers_union = list(set(tickers_c).union(set(tickers_p)))
            
            valid_c = [t for t in tickers_c if t in oos_ret.columns]
            valid_p = [t for t in tickers_p if t in oos_ret.columns]
            valid_union = [t for t in tickers_union if t in oos_ret.columns]
            
            # --- 1. Roda B&H com Custos de Transação para as 3 pontas ---
            ret_c, w_last_c[metric][chave_dict] = calc_buy_hold_return_with_tc(valid_c, oos_ret, w_last_c[metric][chave_dict])
            ret_p, w_last_p[metric][chave_dict] = calc_buy_hold_return_with_tc(valid_p, oos_ret, w_last_p[metric][chave_dict])
            ret_union, w_last_union[metric][chave_dict] = calc_buy_hold_return_with_tc(valid_union, oos_ret, w_last_union[metric][chave_dict])
            
            # --- 2. Regressão do Portfólio Barbell (União) ---
            if ret_union is not None:
                aligned = pd.concat([(ret_union - factors_year['RF']).rename("ExRet"), X_base], axis=1, join="inner").dropna()
                if len(aligned) >= 30:
                    model = sm.OLS(aligned['ExRet'], aligned[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
                    regression_results.append({
                        'metric': metric, 'year_oos': test_year, 'portfolio': f'União {chave_par}',
                        'alpha': model.params['const'] * 252, 'alpha_tstat': model.tvalues['const'],
                        'mkt_beta': model.params['Mkt-RF'], 'smb_beta': model.params['SMB'], 
                        'hml_beta': model.params['HML'], 'r_squared': model.rsquared
                    })
            
            # --- 3. Regressão do Portfólio Spread (Long P_idx Minus Short C_idx) ---
            if ret_c is not None and ret_p is not None:
                spread_ret = ret_p - ret_c # Note: o Long é o P_idx, Short é C_idx
                aligned_spread = pd.concat([spread_ret.rename("ExRet"), X_base], axis=1, join="inner").dropna()
                if len(aligned_spread) >= 30:
                    model_spread = sm.OLS(aligned_spread['ExRet'], aligned_spread[['const', 'Mkt-RF', 'SMB', 'HML']]).fit()
                    regression_results.append({
                        'metric': metric, 'year_oos': test_year, 'portfolio': f'Spread PMC ({chave_par})',
                        'alpha': model_spread.params['const'] * 252, 'alpha_tstat': model_spread.tvalues['const'],
                        'mkt_beta': model_spread.params['Mkt-RF'], 'smb_beta': model_spread.params['SMB'], 
                        'hml_beta': model_spread.params['HML'], 'r_squared': model_spread.rsquared
                    })

df_results = pd.DataFrame(regression_results)

# =========================================================================
# GERAÇÃO DAS TABELAS FAMA-MACBETH
# =========================================================================
def generate_fama_macbeth_table(df_metric_results, metric_name):
    if df_metric_results.empty:
        return None
        
    n_years = df_metric_results['year_oos'].nunique()
    
    summary = df_metric_results.groupby('portfolio').agg({
        'alpha': ['mean', 'std'], 'mkt_beta': 'mean', 'smb_beta': 'mean',
        'hml_beta': 'mean', 'r_squared': 'mean'
    })
    
    summary['Alpha_tstat_FM'] = summary[('alpha', 'mean')] / (summary[('alpha', 'std')] / np.sqrt(n_years))
    summary.columns = ['Alpha', 'Alpha_Std', 'Mkt_Beta', 'SMB_Beta', 'HML_Beta', 'R_Squared', 'Alpha_tstat_FM']
    
    # Organiza os índices na ordem correta
    ordem_pares = [f"D{c}_D{p}" for c, p in decile_pairs]
    ordem_uniao = [f"União {p}" for p in ordem_pares]
    ordem_spread = [f"Spread PMC ({p})" for p in ordem_pares]
    
    todas_ordens = [x for x in ordem_uniao + ordem_spread if x in summary.index]
    summary = summary.reindex(todas_ordens)
    
    final_table = summary[['Alpha', 'Alpha_tstat_FM', 'Mkt_Beta', 'SMB_Beta', 'HML_Beta', 'R_Squared']].round(4)
    
    print(f"\n=== FAMA-MACBETH REGRESSION SUMMARY: {metric_name.upper()} ===")
    display(final_table)
    return final_table

# Executa e exibe as tabelas Fama-MacBeth
fm_hcm = generate_fama_macbeth_table(df_results[df_results['metric'] == 'hcm'], 'hcm')
fm_pozzi = generate_fama_macbeth_table(df_results[df_results['metric'] == 'pozzi'], 'pozzi')

Calculando HCM...
Calculando POZZI...

=== FAMA-MACBETH REGRESSION SUMMARY: HCM ===


,Alpha,Alpha_tstat_FM,Mkt_Beta,SMB_Beta,HML_Beta,R_Squared
portfolio,,,,,,
União D10_D1,0.0375,1.3663,0.5785,0.4047,0.1487,0.8131
União D9_D2,0.0123,0.7808,0.7408,0.4974,0.1479,0.8976
União D8_D3,-0.0112,-0.9783,0.7912,0.4457,0.1783,0.9487
União D7_D4,-0.0010,-0.1358,0.8286,0.4125,0.1712,0.9496
União D6_D5,-0.0040,-0.5932,0.8165,0.4175,0.1960,0.9487
Spread PMC (D10_D1),-0.1253,-2.7059,0.6107,0.3346,0.3958,0.5616
Spread PMC (D9_D2),-0.0302,-0.9095,0.3832,0.0907,0.2841,0.4450
Spread PMC (D8_D3),-0.0018,-0.1056,0.1792,0.1174,0.2672,0.4095
Spread PMC (D7_D4),-0.0331,-1.5616,0.0860,0.1099,0.2244,0.3308



=== FAMA-MACBETH REGRESSION SUMMARY: POZZI ===


,Alpha,Alpha_tstat_FM,Mkt_Beta,SMB_Beta,HML_Beta,R_Squared
portfolio,,,,,,
União D10_D1,0.0271,1.4428,0.6665,0.3827,0.1668,0.8025
União D9_D2,0.0099,0.5352,0.6996,0.3697,0.1792,0.8812
União D8_D3,-0.0154,-1.5567,0.7883,0.4510,0.1857,0.9399
União D7_D4,0.0141,1.7132,0.8063,0.4945,0.1568,0.9423
União D6_D5,-0.0014,-0.0937,0.7931,0.4850,0.1509,0.9242
Spread PMC (D10_D1),-0.0729,-1.7817,0.2288,-0.2599,0.2693,0.3397
Spread PMC (D9_D2),-0.0090,-0.2276,0.1245,-0.1281,0.1854,0.4429
Spread PMC (D8_D3),-0.0014,-0.0624,0.0040,-0.1542,0.0531,0.4945
Spread PMC (D7_D4),0.0117,0.5884,0.0301,-0.1467,0.0151,0.4365
